# 4. Mô hình dữ liệu và validation

## 4.1 Working data và năm bảng

Working publications giữ metadata và trạng thái nguồn theo version. Working references
giữ các cạnh đã thu được; sidecar giữ trạng thái truy vấn. Từ đó M1 xuất hai bảng `papers`
và `references`. M2 tạo `elements`, `hierarchy`, `bibtex` cùng manifest để theo dõi version.

Khóa paper được chuẩn hóa về base ID. Revision vẫn được giữ ở cấu trúc versions và cột
version của hierarchy. Target reference không bắt buộc thuộc dải nhóm; chỉ citing paper
phải có trong papers. Null ở metadata không được chuyển thành một giá trị giả để qua kiểm tra.

## 4.2 Schema

Hai bảng M1 được ghi dưới đây từ thống kê schema đã kiểm tra. Schema M2 được đọc trực tiếp
từ phiên bản source ghim cho các ví dụ thực thi của Book.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
for name, rows in m1['schemas'].items():
    print(name); display(pd.DataFrame(rows))
from parser.storage import SCHEMAS
for name, schema in SCHEMAS.items():
    print(name)
    display(pd.DataFrame([{'column':f.name,'type':str(f.type),'nullable':f.nullable} for f in schema]))

## 4.3 Các invariant được kiểm tra

- `papers`: ID không rỗng, không trùng; metadata và ngày đúng kiểu schema.
- `references`: cặp citing/target hợp lệ, không trùng khóa sau gộp; citing ID thuộc papers.
- `elements`: element ID và paper key hợp lệ, không trùng ID.
- `hierarchy`: node và parent tồn tại trong store của paper, không self-edge hoặc cycle;
  mỗi graph version có đúng một root dạng document.
- `bibtex`: citation key không rỗng, không trùng trong paper; metadata có kiểu đúng.
- Manifest M2: version không trùng, thuộc phạm vi working M1 và khớp run parser; version
  không có TeX chỉ được ghi empty khi M1 đã có bằng chứng tương ứng.

Validator dùng đọc Parquet theo batch và index trên đĩa để đối chiếu khóa xuyên shard.
Cách này giảm nhu cầu giữ full text toàn corpus trong RAM. Đĩa tạm của validation thuộc
phạm vi đo nếu profiler đang theo dõi thư mục chứa nó.

## 4.4 Kết quả trên snapshot

In [ ]:
rows = [{'table':'papers','rows':scope['papers']}, {'table':'references','rows':m1['arxiv_reference_rows']}]
rows += [{'table':key,'rows':value} for key,value in m2['rows'].items()]
display(pd.DataFrame(rows))
display(pd.DataFrame([{'structural_valid':m2['structural_valid'], 'processing_complete':m2['processing_complete'],
                      'expected_versions':scope['versions'], 'recorded_versions':sum(x['recorded_versions'] for x in m2['coverage'])}]))
print('Các kiểm tra cấu trúc đã qua không chứng minh source/reference hoặc parser đều hoàn tất.')

## 4.5 Merge, bảo toàn version và tái lập

Merge dùng base ID rồi hợp nhất theo version tag, tránh chọn một record mới mà bỏ các
version cũ. References dùng các định danh có thể xác minh và giữ xung đột để kiểm tra.
Một shard thiếu stats có thể cung cấp dữ liệu, nhưng không được tự cấp runtime/RAM.

Kết quả hiện tại hợp lệ về cấu trúc trên snapshot. `processing_complete=false` vẫn được
giữ vì có partial/error. Khi nhận dữ liệu mới, cần chạy lại source audit, export và
validation sau parser, thay vì chỉ sửa giá trị boolean trong báo cáo.

## Tài liệu và mã tham chiếu

- [Đặc tả M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/crawler/storage.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/storage.py).
- Mã triển khai: [src/crawler/records.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/records.py).
- Mã triển khai: [src/crawler/merge_shards.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/merge_shards.py).
- Mã triển khai: [src/parser/storage.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/parser/storage.py).
- Mã triển khai: [src/parser/validation.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/parser/validation.py).